# 🚀 Hope ML: IDE-Integrated Training Pipeline

This notebook is optimized for use with the **Antigravity IDE Colab Extension**. It uses a persistent `.env` file on Google Drive for credential management, allowing you to train models without leaving your development environment.

---

In [2]:
# @title ## 1. Hardware & Environment Setup
import torch
import sys
import os

# Check GPU availability
gpu_available = torch.cuda.is_available()
print(f"GPU Available: {gpu_available}")
if gpu_available:
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

# Install core dependencies
!pip install -q torch==2.11.0 pandas==3.0.2 numpy==2.4.4 scikit-learn==1.8.0 tqdm cryptography onnx onnxruntime
print("Dependencies installed.")

GPU Available: True
Device Name: Tesla T4
Dependencies installed.


## 2. Configuration & Secrets

Since you are using the Antigravity IDE extension, we will manage secrets using a `.env` file stored in your Google Drive instead of the Colab Web UI secrets manager.

In [ ]:
# @title ## 2.1 Initialize Configuration File
import os

ENV_PATH = "/content/drive/MyDrive/hope/.env"
EXAMPLE_PATH = "/content/drive/MyDrive/hope/.env.example"

if not os.path.exists(ENV_PATH):
    if os.path.exists(EXAMPLE_PATH):
        print(f"Creating .env from example...")
        !cp {EXAMPLE_PATH} {ENV_PATH}
    else:
        print("Warning: .env.example not found. Creating empty .env.")
        !touch {ENV_PATH}
else:
    print(".env file found in Google Drive.")

In [ ]:
# @title ## 2.2 Credentials Editor
# @markdown Fill in your credentials below and run this cell to save them to the `.env` file.

DERIV_DEMO_TOKEN = "" # @param {type:"string"}
DERIV_REAL_TOKEN = "" # @param {type:"string"}
MODEL_SIGNING_KEY = "" # @param {type:"string"}
SAVE_TO_ENV = True # @param {type:"boolean"}

def update_env(path, updates):
    if not os.path.exists(path): return
    with open(path, "r") as f: lines = f.readlines()
    
    new_lines = []
    keys_seen = set()
    for line in lines:
        matched = False
        for k, v in updates.items():
            if line.startswith(f"{k}="):
                if v: # Only update if not empty
                    new_lines.append(f"{k}={v}\n")
                else:
                    new_lines.append(line)
                keys_seen.add(k)
                matched = True
                break
        if not matched:
            new_lines.append(line)
    
    for k, v in updates.items():
        if k not in keys_seen and v:
            new_lines.append(f"{k}={v}\n")
            
    with open(path, "w") as f: f.writelines(new_lines)
    print(f"Updated {path} with provided credentials.")

if SAVE_TO_ENV:
    updates = {
        "DERIV_DEMO_TOKEN": DERIV_DEMO_TOKEN,
        "DERIV_REAL_TOKEN": DERIV_REAL_TOKEN,
        "MODEL_SIGNING_KEY": MODEL_SIGNING_KEY
    }
    update_env(ENV_PATH, updates)

# Load environment variables into session
with open(ENV_PATH, "r") as f:
    for line in f:
        if "=" in line and not line.startswith("#"):
            k, v = line.strip().split("=", 1)
            os.environ[k] = v
print("Environment variables loaded into current session.")

In [3]:
from google.colab import drive, userdata
import os
import time
import subprocess

# 1. Mount Google Drive
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

REPO_URL = "https://github.com/planetazul3/hope.git"
TARGET_DIR = "/content/drive/MyDrive/hope"

def run_with_backoff(cmd, max_retries=5):
    for i in range(max_retries):
        try:
            subprocess.check_call(cmd, shell=True)
            return True
        except subprocess.CalledProcessError:
            delay = 2 ** i
            print(f"Command failed. Retrying in {delay}s...")
            time.sleep(delay)
    return False

if not os.path.exists(TARGET_DIR):
    print(f"Cloning {REPO_URL} into Google Drive...")
    # Note: Using depth=1 for faster clone in GDrive
    success = run_with_backoff(f"git clone --depth 1 {REPO_URL} {TARGET_DIR}")
else:
    print(f"Pulling latest changes in {TARGET_DIR}...")
    success = run_with_backoff(f"git -C {TARGET_DIR} pull")

if success:
    print("Repository ready in Google Drive.")
    import sys
    scripts_path = os.path.join(TARGET_DIR, "scripts")
    if scripts_path not in sys.path:
        sys.path.append(scripts_path)
else:
    print("Failed to prepare repository.")

# Export secrets to environment variables
try:
    os.environ["MODEL_SIGNING_KEY"] = userdata.get("MODEL_SIGNING_KEY")
    os.environ["DERIV_DEMO_TOKEN"] = userdata.get("DERIV_DEMO_TOKEN")
    os.environ["DERIV_REAL_TOKEN"] = userdata.get("DERIV_REAL_TOKEN")
    os.environ["DERIV_APP_ID"] = "1089" # Default
    print("Secrets loaded into environment.")
except userdata.SecretNotFoundError:
    print("Warning: Secrets not found in Colab userdata. Ensure they are set in the Secrets tab.")


Cloning https://github.com/planetazul3/hope.git into Google Drive...
Repository ready in Google Drive.


TimeoutException: Requesting secret MODEL_SIGNING_KEY timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
# @title ## 3.1 Data Distribution Analysis
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = "/content/drive/MyDrive/hope/data/ticks.csv"
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH, header=None, nrows=10000)
    if df.shape[1] >= 3:
        prices = df.iloc[:, 2]
    else:
        prices = df.iloc[:, 1]
    
    plt.figure(figsize=(10, 4))
    plt.plot(prices)
    plt.title(f"Price Sample from {os.path.basename(CSV_PATH)}")
    plt.xlabel("Ticks")
    plt.ylabel("Price")
    plt.grid(True)
    plt.show()
else:
    print(f"Ticks file not found at {CSV_PATH}. Please run data collection first.")

In [ ]:
# @title ## 4. Training Hyperparameters
BATCH_SIZE = 128 # @param {type:"integer"}
LEARNING_RATE = 0.0001 # @param {type:"number"}
EPOCHS = 100 # @param {type:"integer"}
TRANSFORMER_SEQUENCE_LENGTH = 32 # @param {type:"integer"}
USE_PRETRAINING = True # @param {type:"boolean"}

os.environ["TRANSFORMER_SEQUENCE_LENGTH"] = str(TRANSFORMER_SEQUENCE_LENGTH)
print(f"Configured training for {EPOCHS} epochs with sequence length {TRANSFORMER_SEQUENCE_LENGTH}")

In [ ]:
# @title ## 5. Training & Monitoring
%load_ext tensorboard
import os

# Log directory in GDrive for persistence
LOG_DIR = "/content/drive/MyDrive/hope/logs"
os.makedirs(LOG_DIR, exist_ok=True)

print(f"Launching TensorBoard (logging to {LOG_DIR})...")
%tensorboard --logdir {LOG_DIR}

import train_fixed
train_fixed.main(log_dir=LOG_DIR)